<a href="https://colab.research.google.com/github/Yousra-belg/-tp-csr-rc-auto/blob/main/Copie_de_Untitled4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import os

PROD_PATH = '/content/drive/MyDrive/TP_CSR_RC_Auto/data/Production_B.csv'
SIN_PATH  = '/content/drive/MyDrive/TP_CSR_RC_Auto/data/Sinistre_B.csv'

print("Production :", os.path.exists(PROD_PATH))
print("Sinistres  :", os.path.exists(SIN_PATH))

Production : True
Sinistres  : True


In [5]:
import pandas as pd
import numpy as np

# Chargement des données brutes — on ne les modifie JAMAIS directement
df_prod = pd.read_csv(PROD_PATH)
df_sin  = pd.read_csv(SIN_PATH,  sep=';')
print(f"Production : {df_prod.shape[0]} lignes, {df_prod.shape[1]} colonnes")
print(f"Sinistres  : {df_sin.shape[0]} lignes, {df_sin.shape[1]} colonnes")

Production : 10000 lignes, 10 colonnes
Sinistres  : 5939 lignes, 4 colonnes


In [6]:
df_sin.head()

,numero_police,exercice,code_sinistre,charge
0,18202,2023,33031,3259.13
1,10920,2020,35110,1419.62
2,10754,2019,19586,1188.33
3,16117,2019,36474,2261.72
4,19541,2019,27541,8502.87


In [7]:
df_prod

,numero_police,exercice,Date_naissance,Date_obtention_permis,Date_de_mise_circulation,Kilometrage_parcourus,Sexe,Carburant,Zone,Exposition
0,15890,2018,03/14/1960,10/12/89,06Sep15,890,Man,diesel,Rabat-Sale-Kenitra,0.0309
1,10571,2020,11/03/1991,03/08/15,26Feb05,4543,Woman,essence,Marrakech-Safi,0.2750
2,16417,2019,06/19/1958,09/11/96,09Jul10,2962,Woman,essence,Marrakech-Safi,0.0621
3,21637,2018,06/26/1991,17/03/10,07Sep02,1186,Woman,essence,Casablanca-Settat,0.9888
4,10302,2022,11/22/1952,25/02/79,12Nov11,5779,Man,diesel,Casablanca-Settat,0.2256
...,...,...,...,...,...,...,...,...,...,...
9995,23282,2023,07/22/1953,21/01/08,03Feb12,2450,Woman,essence,Marrakech-Safi,0.7502
9996,29015,2020,06/21/1982,13/01/19,28Oct00,7006,Man,diesel,Casablanca-Settat,0.5120
9997,25243,2020,10/31/1966,04/02/97,29Mar13,3013,Woman,essence,Marrakech-Safi,0.0601
9998,29606,2020,04/01/1985,27/06/12,18May00,372,Woman,essence,CasablancaSettat,0.7650


In [8]:

print("=== PRODUCTION ===")
print(df_prod.dtypes)
print("\nValeurs manquantes :")
print(df_prod.isnull().sum())

print("\n=== SINISTRES ===")
print(df_sin.dtypes)
print("\nValeurs manquantes :")
print(df_sin.isnull().sum())

=== PRODUCTION ===
numero_police                 int64
exercice                      int64
Date_naissance               object
Date_obtention_permis        object
Date_de_mise_circulation     object
Kilometrage_parcourus         int64
Sexe                         object
Carburant                    object
Zone                         object
Exposition                  float64
dtype: object

Valeurs manquantes :
numero_police               0
exercice                    0
Date_naissance              0
Date_obtention_permis       0
Date_de_mise_circulation    0
Kilometrage_parcourus       0
Sexe                        0
Carburant                   0
Zone                        0
Exposition                  0
dtype: int64

=== SINISTRES ===
numero_police      int64
exercice           int64
code_sinistre      int64
charge           float64
dtype: object

Valeurs manquantes :
numero_police    0
exercice         0
code_sinistre    0
charge           0
dtype: int64


In [9]:
#on corrige l'exposition
mask = (df_prod['Exposition'] > 1) | (df_prod['Exposition'] < 0)
print(f"Lignes hors-bornes : {mask.sum()}")
print(df_prod[mask][['numero_police', 'exercice', 'Exposition']])

Lignes hors-bornes : 6
      numero_police  exercice  Exposition
66            25690      2018     -0.5582
2117          23116      2020      1.2000
5196          15080      2019     -0.2629
6612          19818      2022     -0.5187
8879          27773      2019      1.2000
9202          22343      2020      1.2000


In [10]:
#clip remplace les valeurs hors-bornes par la borne la plus proche
df_prod['Exposition'] = df_prod['Exposition'].clip(0, 1)

print("\nAprès correction :")
print( df_prod[mask]['Exposition'])



Après correction :
66      0.0
2117    1.0
5196    0.0
6612    0.0
8879    1.0
9202    1.0
Name: Exposition, dtype: float64


In [11]:
print(df_prod['Sexe'].value_counts().to_dict())

{'Woman': 5757, 'Man': 4152, 'G': 48, ' ': 43}


In [12]:
 # Nettoyage de la variable Sexe
df_prod['Sexe'] = df_prod['Sexe'].str.strip().str.upper()
df_prod['Sexe'] = df_prod['Sexe'].replace({
    'M'   : 'Homme',
    'F'   : 'Femme',
    'H'   : 'Homme',
    'G'   : 'Inconnu',   # ← G est traité ici par replace
    ''    : 'Inconnu',   # ← espace vide est traité ici par replace
})
df_prod['Sexe'] = df_prod['Sexe'].fillna('Inconnu')  # ← traite les NaN
print(df_prod['Sexe'].value_counts().to_dict())

{'WOMAN': 5757, 'MAN': 4152, 'Inconnu': 91}


In [13]:
# Voir toutes les modalités exactes de Zone
for zone in sorted(df_prod['Zone'].unique()):
    print(repr(zone))

'BeniMellal-Khenifra'
'Casablanca-Settat'
'CasablancaSettat'
'Dakhla-OuedEddahab'
'Draa-Tafilalet'
'Fes-Meknes'
'Guelmim'
'Laayoune-SakiaElHamra'
'Marrakech-Safi'
'Oriental'
'Rabat-Sale-Kenitra'
'Souss-Massa'
'Tanger-Tetouan-AlHoceima'
'TangerTetouanAlHoceima'


In [14]:
mapping_zones = {
    'CasablancaSettat'       : 'Casablanca-Settat',
    'Tanger-Tetouan-Al Hoceima' : 'TangerTetouanAlHoceima',
}
df_prod['Zone'] = df_prod['Zone'].replace(mapping_zones)



In [15]:
for zone in sorted(df_prod['Zone'].unique()):
    print(repr(zone))

'BeniMellal-Khenifra'
'Casablanca-Settat'
'Dakhla-OuedEddahab'
'Draa-Tafilalet'
'Fes-Meknes'
'Guelmim'
'Laayoune-SakiaElHamra'
'Marrakech-Safi'
'Oriental'
'Rabat-Sale-Kenitra'
'Souss-Massa'
'Tanger-Tetouan-AlHoceima'
'TangerTetouanAlHoceima'


In [16]:
# ── Correction du format Date_mise_circulation ─────────
df_prod['Date_de_mise_circulation'] = pd.to_datetime(
    df_prod['Date_de_mise_circulation'], format='%d%b%y', errors='coerce'
)

# ── Correction du format Date_obtention_permis ──────────
df_prod['Date_obtention_permis'] = pd.to_datetime(
    df_prod['Date_obtention_permis'], format='%d/%m/%y', errors='coerce'
)

# ── Correction du format Date_naissance ─────────────────
df_prod['Date_naissance'] = pd.to_datetime(
    df_prod['Date_naissance'], format='%m/%d/%Y', errors='coerce'
)

# Vérification
print(df_prod[['Date_naissance','Date_obtention_permis','Date_de_mise_circulation']].head(5))

  Date_naissance Date_obtention_permis Date_de_mise_circulation
0     1960-03-14            1989-12-10               2015-09-06
1     1991-11-03            2015-08-03               2005-02-26
2     1958-06-19            1996-11-09               2010-07-09
3     1991-06-26            2010-03-17               2002-09-07
4     1952-11-22            1979-02-25               2011-11-12


In [18]:
# Âge du conducteur (en années)
ref = pd.Timestamp('2023-12-31')  # date de référence = fin de portefeuille
df_prod['age_conducteur'] = ((ref - df_prod['Date_naissance']).dt.days / 365.25).round(1)

# Ancienneté du permis (en années)
df_prod['anciennete_permis'] = ((ref - df_prod['Date_obtention_permis']).dt.days / 365.25).round(1)

# Âge du véhicule (en années)
df_prod['age_vehicule'] = ((ref - df_prod['Date_de_mise_circulation']).dt.days / 365.25).round(1)

print(df_prod[['age_conducteur','anciennete_permis','age_vehicule']].describe().round(2))

       age_conducteur  anciennete_permis  age_vehicule
count        10000.00           10000.00      10000.00
mean            48.65              17.23         15.53
std             14.59              11.65          4.90
min             21.00             -44.90          7.00
25%             36.20               8.00         11.28
50%             48.60              14.25         15.60
75%             61.10              24.10         19.80
max             74.00              54.60         24.00


In [19]:
# Diagnostic - ancienneté négative ou aberrante
print("Ancienneté négative :")
print(df_prod[df_prod['anciennete_permis'] < 0][['numero_police', 'Date_obtention_permis', 'anciennete_permis']])

print(f"\nNombre de lignes concernées : {(df_prod['anciennete_permis'] < 0).sum()}")

Ancienneté négative :
      numero_police Date_obtention_permis  anciennete_permis
563           14645            2068-11-11              -44.9
2519          17389            2068-10-01              -44.8
2585          29841            2068-03-06              -44.2
4408          17456            2068-03-27              -44.2
5329          28811            2068-09-25              -44.7
6576          20155            2068-01-21              -44.1

Nombre de lignes concernées : 6


In [20]:
# Vérifier la cohérence avec Date_naissance avant de corriger
mask_2068 = df_prod['Date_obtention_permis'].dt.year == 2068

verification = df_prod[mask_2068][['numero_police',
                                    'Date_naissance',
                                    'Date_obtention_permis']].copy()

# Si on soustrait 100 ans, quel serait l'âge au permis ?
verification['permis_corrige'] = (
    df_prod.loc[mask_2068, 'Date_obtention_permis'] - pd.DateOffset(years=100)
)
verification['age_au_permis_si_correction'] = (
    (verification['permis_corrige'] - verification['Date_naissance']).dt.days / 365.25
).round(1)

print(verification)

      numero_police Date_naissance Date_obtention_permis permis_corrige  \
563           14645     1950-10-15            2068-11-11     1968-11-11   
2519          17389     1950-09-23            2068-10-01     1968-10-01   
2585          29841     1950-11-09            2068-03-06     1968-03-06   
4408          17456     1950-06-07            2068-03-27     1968-03-27   
5329          28811     1950-10-18            2068-09-25     1968-09-25   
6576          20155     1950-11-28            2068-01-21     1968-01-21   

      age_au_permis_si_correction  
563                          18.1  
2519                         18.0  
2585                         17.3  
4408                         17.8  
5329                         17.9  
6576                         17.1  


In [21]:
# Correction validée — on applique
df_prod.loc[mask_2068, 'Date_obtention_permis'] = (
    df_prod.loc[mask_2068, 'Date_obtention_permis'] - pd.DateOffset(years=100)
)

# Recalcul de l'ancienneté
df_prod['anciennete_permis'] = (
    (ref - df_prod['Date_obtention_permis']).dt.days / 365.25
).round(1)

# Vérification finale
print("✓ Correction appliquée")
print(df_prod['anciennete_permis'].describe().round(2))
print(f"\nAnciennetés négatives restantes : {(df_prod['anciennete_permis'] < 0).sum()}")

✓ Correction appliquée
count    10000.00
mean        17.29
std         11.59
min          1.00
25%          8.00
50%         14.30
75%         24.20
max         55.90
Name: anciennete_permis, dtype: float64

Anciennetés négatives restantes : 0


In [22]:
print(df_prod[['age_conducteur','anciennete_permis','age_vehicule']].describe().round(2))

       age_conducteur  anciennete_permis  age_vehicule
count        10000.00           10000.00      10000.00
mean            48.65              17.29         15.53
std             14.59              11.59          4.90
min             21.00               1.00          7.00
25%             36.20               8.00         11.28
50%             48.60              14.30         15.60
75%             61.10              24.20         19.80
max             74.00              55.90         24.00


In [23]:
print(df_prod[['age_conducteur','anciennete_permis','age_vehicule']])

      age_conducteur  anciennete_permis  age_vehicule
0               63.8               34.1           8.3
1               32.2                8.4          18.8
2               65.5               27.1          13.5
3               32.5               13.8          21.3
4               71.1               44.8          12.1
...              ...                ...           ...
9995            70.4               15.9          11.9
9996            41.5                5.0          23.2
9997            57.2               26.9          10.8
9998            38.7               11.5          23.6
9999            61.0               17.0           9.1

[10000 rows x 3 columns]


In [24]:
# ── Diagnostic complet de df_sin ─────────────────────
print("=== STRUCTURE ===")
print(df_sin.dtypes)
print(f"\nLignes : {df_sin.shape[0]}, Colonnes : {df_sin.shape[1]}")

print("\n=== VALEURS MANQUANTES ===")
print(df_sin.isnull().sum())

print("\n=== APERÇU ===")
print(df_sin.head(10))

print("\n=== STATISTIQUES CHARGE ===")
print(df_sin['charge'].describe().round(2))

print("\n=== VALEURS ABERRANTES CHARGE ===")
print(f"Charges nulles (= 0)   : {(df_sin['charge'] == 0).sum()}")
print(f"Charges négatives      : {(df_sin['charge'] < 0).sum()}")
print(f"Charges > 100 000 MAD  : {(df_sin['charge'] > 100000).sum()}")

print("\n=== EXERCICES PRÉSENTS ===")
print(df_sin['exercice'].value_counts().sort_index())

print("\n=== DOUBLONS ===")
print(f"Lignes dupliquées : {df_sin.duplicated().sum()}")
print(f"Même code_sinistre en double : {df_sin.duplicated(subset='code_sinistre').sum()}")

=== STRUCTURE ===
numero_police      int64
exercice           int64
code_sinistre      int64
charge           float64
dtype: object

Lignes : 5939, Colonnes : 4

=== VALEURS MANQUANTES ===
numero_police    0
exercice         0
code_sinistre    0
charge           0
dtype: int64

=== APERÇU ===
   numero_police  exercice  code_sinistre   charge
0          18202      2023          33031  3259.13
1          10920      2020          35110  1419.62
2          10754      2019          19586  1188.33
3          16117      2019          36474  2261.72
4          19541      2019          27541  8502.87
5          24133      2023          25177  1482.54
6          17527      2019          28116  2185.05
7          22737      2019          19942  1269.50
8          27883      2023          16138  3313.69
9          19233      2020          22138  1306.26

=== STATISTIQUES CHARGE ===
count     5939.00
mean      2271.22
std       1301.11
min        133.98
25%       1330.28
50%       1998.40
75%     

In [25]:
# Identifier le doublon
doublon = df_sin[df_sin.duplicated(subset='code_sinistre', keep=False)]
print("Sinistre en double :")
print(doublon)

Sinistre en double :
      numero_police  exercice  code_sinistre   charge
5935            883      2019            146  1009.24
5938           4691      2021            146   998.85


In [26]:
import os

# Chemin de sauvegarde
PROCESSED_PATH = '/content/drive/MyDrive/TP_CSR_RC_Auto/data/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Sauvegarde
df_prod.to_csv(PROCESSED_PATH + 'Production_nettoyee.csv', index=False)
df_sin.to_csv(PROCESSED_PATH  + 'Sinistre_nettoye.csv',   index=False)

print("Fichiers sauvegardés :")
print("  ✓ Production_nettoyee.csv")
print("  ✓ Sinistre_nettoye.csv")

Fichiers sauvegardés :
  ✓ Production_nettoyee.csv
  ✓ Sinistre_nettoye.csv


In [28]:
# Chargement de la base nettoyée
OUTPUT_PATH = '/content/drive/MyDrive/TP_CSR_RC_Auto/data/'
df_prod_net = pd.read_csv(OUTPUT_PATH + 'Production_nettoyee.csv')

print("=== BILAN COMPLET BASE NETTOYÉE ===")
print(f"Lignes : {df_prod_net.shape[0]}, Colonnes : {df_prod_net.shape[1]}")
print(f"\nColonnes : {list(df_prod_net.columns)}")

print("\n=== VALEURS MANQUANTES ===")
print(df_prod_net.isnull().sum())

print("\n=== VARIABLES CATÉGORIELLES ===")
print("Sexe :", df_prod_net['Sexe'].value_counts().to_dict())
print("Carburant :", df_prod_net['Carburant'].value_counts().to_dict())
print("Zone - nb modalités :", df_prod_net['Zone'].nunique())
print(df_prod_net['Zone'].value_counts())

print("\n=== VARIABLES NUMÉRIQUES ===")
cols_num = ['Exposition','Kilometrage_parcourus','age_conducteur','anciennete_permis','age_vehicule']
print(df_prod_net[cols_num].describe().round(2))

=== BILAN COMPLET BASE NETTOYÉE ===
Lignes : 10000, Colonnes : 13

Colonnes : ['numero_police', 'exercice', 'Date_naissance', 'Date_obtention_permis', 'Date_de_mise_circulation', 'Kilometrage_parcourus', 'Sexe', 'Carburant', 'Zone', 'Exposition', 'age_conducteur', 'anciennete_permis', 'age_vehicule']

=== VALEURS MANQUANTES ===
numero_police               0
exercice                    0
Date_naissance              0
Date_obtention_permis       0
Date_de_mise_circulation    0
Kilometrage_parcourus       0
Sexe                        0
Carburant                   0
Zone                        0
Exposition                  0
age_conducteur              0
anciennete_permis           0
age_vehicule                0
dtype: int64

=== VARIABLES CATÉGORIELLES ===
Sexe : {'WOMAN': 5757, 'MAN': 4152, 'Inconnu': 91}
Carburant : {'diesel': 5367, 'essence': 4529, ' ': 104}
Zone - nb modalités : 13
Zone
Casablanca-Settat           3389
Marrakech-Safi              2173
Tanger-Tetouan-AlHoceima    212

In [29]:
# Voir toutes les modalités exactes
for zone in sorted(df_prod_net['Zone'].unique()):
    print(repr(zone))

'BeniMellal-Khenifra'
'Casablanca-Settat'
'Dakhla-OuedEddahab'
'Draa-Tafilalet'
'Fes-Meknes'
'Guelmim'
'Laayoune-SakiaElHamra'
'Marrakech-Safi'
'Oriental'
'Rabat-Sale-Kenitra'
'Souss-Massa'
'Tanger-Tetouan-AlHoceima'
'TangerTetouanAlHoceima'


In [30]:
# Correction unique Zone
df_prod_net['Zone'] = df_prod_net['Zone'].replace({
    'TangerTetouanAlHoceima' : 'Tanger-Tetouan-AlHoceima'
})

# Vérification
print("Modalités après correction :", df_prod_net['Zone'].nunique())
print(df_prod_net['Zone'].value_counts())

Modalités après correction : 12
Zone
Casablanca-Settat           3389
Tanger-Tetouan-AlHoceima    2360
Marrakech-Safi              2173
Rabat-Sale-Kenitra           267
Guelmim                      260
Dakhla-OuedEddahab           239
Fes-Meknes                   229
Draa-Tafilalet               227
Oriental                     223
BeniMellal-Khenifra          222
Souss-Massa                  213
Laayoune-SakiaElHamra        198
Name: count, dtype: int64


In [32]:
# Chargement sinistres
df_sin = pd.read_csv(SIN_PATH, sep=';')

# Agrégation par police-exercice
df_freq = df_sin.groupby(['numero_police','exercice']).agg(
    nb_sinistres  = ('code_sinistre', 'count'),
    charge_totale = ('charge', 'sum')
).reset_index()

# Jointure
df_final = df_prod_net.merge(df_freq, on=['numero_police','exercice'], how='left')
df_final['nb_sinistres']  = df_final['nb_sinistres'].fillna(0).astype(int)
df_final['charge_totale'] = df_final['charge_totale'].fillna(0)

# Sauvegarde
df_final.to_csv(OUTPUT_PATH + 'Base_finale_membre2.csv', index=False)
print(f"✓ Fichier livré : {df_final.shape[0]} lignes, {df_final.shape[1]} colonnes")

✓ Fichier livré : 10000 lignes, 15 colonnes
